In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
from insightface.app import FaceAnalysis

In [2]:
db_dir = Path('DB')
face = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
face.prepare(ctx_id=0, det_size=(1280, 1280))

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Corei5_8GBRAM_512SSD/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Corei5_8GBRAM_512SSD/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Corei5_8GBRAM_512SSD/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Corei5_8GBRAM_512SSD/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Corei5_8GBRAM_512SSD/.insight

In [3]:
database = {}
skipped_files = []

for image_path in db_dir.iterdir():
    if not image_path.is_file():
        continue

    img = cv2.imread(str(image_path))
    faces = face.get(img)

    if not faces:
        skipped_files.append(image_path.name)
        continue

    best_face = max(faces, key=lambda detected_face: detected_face.det_score)
    database[image_path.stem] = best_face.embedding.astype(np.float32)

print(f'Stored embeddings for {len(database)} images')
print(f'Skipped {len(skipped_files)} images with no detected face')

C:\Users\Corei5_8GBRAM_512SSD\AppData\Roaming\Python\Python312\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


Stored embeddings for 16 images
Skipped 6 images with no detected face


In [4]:
sample_key = next(iter(database))
print(sample_key)
print(type(database[sample_key]))
print(database[sample_key].shape)

Srishti Chamoli,23BCS111
<class 'numpy.ndarray'>
(512,)


### Save the database as .json file

In [ ]:
# json_path = Path('embeddings.json')
# json_ready_database = {name: embedding.tolist() for name, embedding in database.items()}

# with json_path.open('w', encoding='utf-8') as json_file:
#     json.dump(json_ready_database, json_file, indent=2)

# print(f'Saved embeddings to {json_path.resolve()}')

### Save the database as a .npz file

In [6]:
np.savez('embeddings.npz', names=list(database.keys()), embeddings=np.array(list(database.values())))